# 02 - Global Trends and Time Series Analysis
7-day averages, wave detection, decomposition, and forecasting baselines.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import STL
from src.covid_analyzer import COVIDAnalyzer
plt.style.use('seaborn-v0_8')
sns.set_context('talk')

In [ ]:
an = COVIDAnalyzer()
_ = an.load_data(True)
GLOBAL, waves = an.analyze_global_trends()
GLOBAL.head()

## Global 7-day Averages and Waves

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(14,8))
ax[0].plot(GLOBAL['date'], GLOBAL['cases_7day_avg'], label='Cases 7d')
ax[0].set_title('Global Daily Cases (7d)')
ax[1].plot(GLOBAL['date'], GLOBAL['deaths_7day_avg'], color='r', label='Deaths 7d')
ax[1].set_title('Global Daily Deaths (7d)')
for w in waves:
    for a in ax:
        a.axvline(pd.to_datetime(w['peak_date']), color='k', ls='--', alpha=0.2)
plt.tight_layout()

## Decomposition (STL) of Cases

In [ ]:
series = GLOBAL.set_index('date')['cases_7day_avg'].interpolate().asfreq('D')
stl = STL(series, period=91).fit()
fig = stl.plot()

## Baseline Forecast (Naive and Moving Average)

In [ ]:
h=30
naive_forecast = series.shift(7).iloc[-h:]
ma_forecast = series.rolling(14).mean().iloc[-h:]
ax = series[-200:].plot(figsize=(14,5), label='Actual')
naive_forecast.index = series.index[-h:]
naive_forecast.plot(ax=ax, label='Naive-7')
ma_forecast.index = series.index[-h:]
ma_forecast.plot(ax=ax, label='MA-14')
plt.legend(); plt.title('Baseline Forecasts (last 200 days)')